In [21]:
from pathlib import Path

KNOWLEDGE_PATH = Path("data/knowledge.txt")

text = KNOWLEDGE_PATH.read_text(
    encoding="utf-8"
)

print(f"Loaded {len(text)} characters")
print(text[:500])

Loaded 898 characters
Machine learning is a field of artificial intelligence in which computers learn patterns from data and use those patterns to make predictions or decisions.

Supervised learning uses labelled examples, while unsupervised learning finds patterns in data without labels.

An algorithm is a finite sequence of precise steps for solving a problem.

A data structure is a way to organise data so that operations such as lookup, insertion, and deletion can be performed efficiently.

Retrieval-augmented gen


In [22]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80
)

chunks = splitter.create_documents([text])

print("Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks[:5]):
    print(f"\n--- Chunk {i + 1} ---")
    print(chunk.page_content)

Number of chunks: 2

--- Chunk 1 ---
Machine learning is a field of artificial intelligence in which computers learn patterns from data and use those patterns to make predictions or decisions.

Supervised learning uses labelled examples, while unsupervised learning finds patterns in data without labels.

An algorithm is a finite sequence of precise steps for solving a problem.

A data structure is a way to organise data so that operations such as lookup, insertion, and deletion can be performed efficiently.

--- Chunk 2 ---
Retrieval-augmented generation (RAG) retrieves relevant source text before asking a language model to answer.

An embedding is a numerical vector that represents the meaning of text, an image, or another object. Items with similar meanings have vectors that are close together.

In a RAG system, document embeddings are stored in a vector database and a question embedding is used to retrieve the most relevant documents.


In [23]:
import os
from dotenv import load_dotenv

load_dotenv()

from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=os.getenv("GEMINI_API_KEY")
)

print("Embedding model loaded.")

Embedding model loaded.


In [24]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

print("FAISS index created.")

FAISS index created.


In [25]:
query = "What is machine learning?"

results = vectorstore.similarity_search(
    query,
    k=3
)

for i, result in enumerate(results):

    print(f"\n--- Result {i + 1} ---")
    print(result.page_content)


--- Result 1 ---
Machine learning is a field of artificial intelligence in which computers learn patterns from data and use those patterns to make predictions or decisions.

Supervised learning uses labelled examples, while unsupervised learning finds patterns in data without labels.

An algorithm is a finite sequence of precise steps for solving a problem.

A data structure is a way to organise data so that operations such as lookup, insertion, and deletion can be performed efficiently.

--- Result 2 ---
Retrieval-augmented generation (RAG) retrieves relevant source text before asking a language model to answer.

An embedding is a numerical vector that represents the meaning of text, an image, or another object. Items with similar meanings have vectors that are close together.

In a RAG system, document embeddings are stored in a vector database and a question embedding is used to retrieve the most relevant documents.


In [26]:
query = "How do computers learn patterns from data?"

results = vectorstore.similarity_search(
    query,
    k=3
)

for i, result in enumerate(results):

    print(f"\n--- Result {i + 1} ---")
    print(result.page_content)


--- Result 1 ---
Machine learning is a field of artificial intelligence in which computers learn patterns from data and use those patterns to make predictions or decisions.

Supervised learning uses labelled examples, while unsupervised learning finds patterns in data without labels.

An algorithm is a finite sequence of precise steps for solving a problem.

A data structure is a way to organise data so that operations such as lookup, insertion, and deletion can be performed efficiently.

--- Result 2 ---
Retrieval-augmented generation (RAG) retrieves relevant source text before asking a language model to answer.

An embedding is a numerical vector that represents the meaning of text, an image, or another object. Items with similar meanings have vectors that are close together.

In a RAG system, document embeddings are stored in a vector database and a question embedding is used to retrieve the most relevant documents.


In [27]:
VECTORSTORE_PATH = "vectorstore"

vectorstore.save_local(
    VECTORSTORE_PATH
)

print(
    f"FAISS vector store saved to {VECTORSTORE_PATH}"
)

FAISS vector store saved to vectorstore


In [28]:
evaluation_data = [
    {
        "question": "What is machine learning?",
        "expected_keyword": "machine learning"
    },
    {
        "question": "What is supervised learning?",
        "expected_keyword": "supervised"
    },
    {
        "question": "What is an embedding?",
        "expected_keyword": "embedding"
    },
    {
        "question": "What is RAG?",
        "expected_keyword": "retrieval"
    },
]

In [29]:
correct = 0

for item in evaluation_data:

    results = vectorstore.similarity_search(
        item["question"],
        k=1
    )

    retrieved_text = results[0].page_content.lower()

    if item["expected_keyword"] in retrieved_text:
        correct += 1

retrieval_accuracy = correct / len(
    evaluation_data
)

print(
    f"Retrieval Accuracy: "
    f"{retrieval_accuracy:.2%}"
)

Retrieval Accuracy: 100.00%
